In [1]:
from google.colab import files
uploaded = files.upload()

Saving Online Retail.csv to Online Retail.csv


In [2]:
import pandas as pd

df = pd.read_csv('Online Retail.csv', encoding="latin1")
df_raw = df.copy()

print(df.shape)
print(df.dtypes)
print(df.head())
print(df.isnull().sum())

(541909, 8)
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

     InvoiceDate  UnitPrice  CustomerID         Country  
0  01-12-10 8:26       2.55     17850.0  United Kingdom  
1  01-12-10 8:26       3.39     17850.0  United Kingdom  
2  01-12-10 8:26       2.75     17850.0  United Kingdom  
3  01-12-10 8:26       3.39     17850.0  United Kingdom  
4  01-12-10 8:26       3.39     17850.0  United Kingdom  
InvoiceNo

In [3]:
# Step 1: Drop rows where Description or CustomerID are missing
df = df.dropna(subset=["Description", "CustomerID"])

# Step 2: Remove cancellations
# InvoiceNo starts with "C" for cancelled orders -- we want to keep only normal ones
cancelled = df["InvoiceNo"].astype(str).str.startswith("C")
df = df[cancelled == False]

# Step 3: Keep only rows where Quantity and UnitPrice are greater than zero
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# Step 4: Fix data types
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)

# Step 5: Add a Revenue column
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Check the result
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

/tmp/ipykernel_1700/1968948881.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


(397884, 9)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
Revenue               float64
dtype: object
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
dtype: int64


In [4]:
#135,080 rows were removed due to missing CustomerID values, likely representing guest transactions.
#These were excluded from customer-level analysis to maintain accuracy.

In [5]:
# Check date range
print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

# Check unique counts
print(df["Country"].value_counts())
print(df["CustomerID"].nunique())

2010-01-12 08:26:00
2011-12-10 17:19:00
Country
United Kingdom          354321
Germany                   9040
France                    8341
EIRE                      7236
Spain                     2484
Netherlands               2359
Belgium                   2031
Switzerland               1841
Portugal                  1462
Australia                 1182
Norway                    1071
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     248
Unspecified                244
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                   

In [6]:
#Q1 -- How did monthly revenue trend over the year?

# Extract month and year from InvoiceDate into a new column
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")

# Group by YearMonth and sum the Revenue
monthly_revenue = df.groupby("YearMonth")["Revenue"].sum()

print(monthly_revenue)


# Revenue was low and inconsistent through most of 2010 (incomplete data)
# 2011 shows a clear growth trend peaking in September and November
# December 2011 drops sharply -- partial month, data cuts off Dec 10th

YearMonth
2010-01      46376.490
2010-02      47316.530
2010-03      23921.710
2010-05      31771.600
2010-06      31215.640
2010-07      53795.310
2010-08      39248.820
2010-09      38231.900
2010-10      33650.280
2010-12     227185.610
2011-01     601804.600
2011-02     487100.630
2011-03     646325.520
2011-04     602872.241
2011-05     719269.560
2011-06     688561.560
2011-07     735451.041
2011-08     608723.700
2011-09    1095156.872
2011-10     904826.420
2011-11    1043411.070
2011-12     205190.800
Freq: M, Name: Revenue, dtype: float64


In [7]:
# Checking the minimum and maximum date within each month to see the actual range of days covered.

df.groupby("YearMonth")["InvoiceDate"].agg(["min", "max"])

# As observed, The 2010 data is essentially unusable for trend analysis, a single day of transactions isn't representative of anything meaningful.

,min,max
YearMonth,,
2010-01,2010-01-12 08:26:00,2010-01-12 17:35:00
2010-02,2010-02-12 07:48:00,2010-02-12 19:59:00
2010-03,2010-03-12 09:31:00,2010-03-12 17:28:00
2010-05,2010-05-12 10:03:00,2010-05-12 16:41:00
2010-06,2010-06-12 08:34:00,2010-06-12 17:26:00
2010-07,2010-07-12 09:13:00,2010-07-12 17:19:00
2010-08,2010-08-12 08:12:00,2010-08-12 17:20:00
2010-09,2010-09-12 08:34:00,2010-09-12 20:01:00
2010-10,2010-10-12 09:33:00,2010-10-12 17:26:00


In [8]:
#Q2 -- Which months had the highest and lowest sales?


monthly_revenue.sort_values(ascending = False).head(5)
monthly_revenue.sort_values().head(5)

# 2010 data is incomplete, with some months having only a single day of transactions. As such, lowest sales info can be rendered unusable for 2010.

,Revenue
YearMonth,
2010-03,23921.71
2010-06,31215.64
2010-05,31771.60
2010-10,33650.28
2010-09,38231.90


In [9]:
#Q3 - What does weekly order volume look like?

# Extract month and year from InvoiceDate into a new column
df["YearWeek"] = df["InvoiceDate"].dt.to_period("W")

# Group by YearMonth and sum the Revenue
weekly_orders = df.groupby("YearWeek")["InvoiceNo"].nunique()

print(weekly_orders)


# Weekly order volume grows significantly from 2010 to 2011, consistent with what we saw in the monthly revenue trend
# The busiest weeks cluster around November 2011, peaking at 669 unique orders in the week of Nov 14-20
# The 2010 gaps are visible here too -- missing weeks confirm the incomplete data picture

YearWeek
2010-01-11/2010-01-17    121
2010-02-08/2010-02-14    137
2010-03-08/2010-03-14     57
2010-05-10/2010-05-16     87
2010-06-07/2010-06-13     94
                        ... 
2011-11-07/2011-11-13    441
2011-11-14/2011-11-20    669
2011-11-21/2011-11-27    594
2011-11-28/2011-12-04    441
2011-12-05/2011-12-11    362
Freq: W-SUN, Name: InvoiceNo, Length: 62, dtype: int64


In [10]:
#Q4 - What are the top 10 best-selling products by revenue?

df.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)


# Note - Postage and Manual are administrative entries and essentially non-product revenue.

,Revenue
Description,
"PAPER CRAFT , LITTLE BIRDIE",168469.60
REGENCY CAKESTAND 3 TIER,142592.95
WHITE HANGING HEART T-LIGHT HOLDER,100448.15
JUMBO BAG RED RETROSPOT,85220.78
MEDIUM CERAMIC TOP STORAGE JAR,81416.73
POSTAGE,77803.96
PARTY BUNTING,68844.33
ASSORTED COLOUR BIRD ORNAMENT,56580.34
Manual,53779.93


In [11]:
#Q5 - Which products are most frequently cancelled?

cancelled = df_raw["InvoiceNo"].astype(str).str.startswith("C")
df_cancelled = df_raw[cancelled]

df_cancelled.groupby("Description")["InvoiceNo"].count().sort_values(ascending=False).head(10)

# The standout observation here is REGENCY CAKESTAND 3 TIER. It is the second-best selling item by revenue but also the second most cancelled product.

,InvoiceNo
Description,
Manual,244
REGENCY CAKESTAND 3 TIER,181
POSTAGE,126
JAM MAKING SET WITH JARS,87
Discount,77
SET OF 3 CAKE TINS PANTRY DESIGN,74
SAMPLES,61
STRAWBERRY CERAMIC TRINKET BOX,55
ROSES REGENCY TEACUP AND SAUCER,54


In [12]:
#Q6 - How many unique customers made purchases, and what is the average order value?

print(df["CustomerID"].nunique())

total_revenue = df.groupby('InvoiceNo')['Revenue'].sum()
aov = total_revenue.mean()

print(aov)

4338
480.86595639974104


In [13]:
#Q7 - Who are the top customers by total spend?

df.groupby('CustomerID')['Revenue'].sum().sort_values(ascending=False).head(10)

,Revenue
CustomerID,
14646,280206.02
18102,259657.30
17450,194550.79
16446,168472.50
14911,143825.06
12415,124914.53
14156,117379.63
17511,91062.38
16029,81024.84


In [14]:
#Q8 - What percentage of revenue comes from repeat vs one-time customers?

order_counts = df.groupby('CustomerID')['InvoiceNo'].nunique()
is_repeat = order_counts > 1

print(pd.value_counts(is_repeat))

df['IsRepeat'] = df['CustomerID'].map(is_repeat)
revenue_by_type = df.groupby('IsRepeat')['Revenue'].sum()
(revenue_by_type / revenue_by_type.sum()) * 100

# Repeat customers (65%) generate 93% of total revenue
# One-time customers (35%) generate only 7% of revenue
# The business is heavily reliant on repeat customers and so losing even a small number of them would have big revenue impact.

InvoiceNo
True     2845
False    1493
Name: count, dtype: int64


/tmp/ipykernel_1700/271197328.py:6: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(is_repeat))


,Revenue
IsRepeat,
False,6.915986
True,93.084014


In [15]:
#Q9 - Which countries generate the most revenue outside the UK?

df_international = df[df["Country"] != "United Kingdom"]
df_international.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

,Revenue
Country,
Netherlands,285446.34
EIRE,265545.90
Germany,228867.14
France,209024.05
Australia,138521.31
Spain,61577.11
Switzerland,56443.95
Belgium,41196.34
Sweden,38378.33


In [16]:
#Q10 - How does average order value differ by country?

order_totals = df.groupby(['InvoiceNo', 'Country'])['Revenue'].sum()
aov_by_country = order_totals.groupby('Country').mean()

print(aov_by_country)

Country
Australia               2430.198421
Austria                  599.922353
Bahrain                  274.200000
Belgium                  420.370816
Brazil                  1143.600000
Canada                   611.063333
Channel Islands          786.555385
Cyprus                   849.398750
Czech Republic           413.370000
Denmark                 1053.074444
EIRE                    1021.330385
European Community       325.062500
Finland                  549.904390
France                   537.336889
Germany                  500.803370
Greece                   952.104000
Iceland                  615.714286
Israel                  1444.338000
Italy                    460.085263
Japan                   1969.282632
Lebanon                 1693.880000
Lithuania                415.265000
Malta                    545.118000
Netherlands             3036.663191
Norway                  1004.595556
Poland                   386.034211
Portugal                 586.664737
RSA                 

In [17]:
df.to_csv("online_retail_cleaned.csv", index=False)

In [18]:
#Q11 - How many new customers are acquired each month?

first_purchase = df.groupby('CustomerID')['InvoiceDate'].min()
new_customers_monthly = first_purchase.dt.to_period('M').value_counts().sort_index()

print(new_customers_monthly)

# 2010 numbers are low but that's the incomplete data issue
# January 2011 was the biggest month for new customer acquisition at 544
# There's a clear declining trend through 2011, from 544 in January down to 230 by November
# December 2011 shows only 31, again due partial month
# New customer acquisition is declining month over month through 2011. That's a warning sign for the business.
# It means the customer base is increasingly reliant on existing customers rather than growth.

InvoiceDate
2010-01     95
2010-02     93
2010-03     46
2010-05     69
2010-06     70
2010-07     50
2010-08     83
2010-09     67
2010-10     40
2010-12    272
2011-01    544
2011-02    424
2011-03    416
2011-04    318
2011-05    280
2011-06    270
2011-07    233
2011-08    179
2011-09    256
2011-10    272
2011-11    230
2011-12     31
Freq: M, Name: count, dtype: int64


In [19]:
#Q12 - What is the monthly retention rate of customers? (Of the customers who bought in month X, how many came back in month X+1?)

customer_months = df[['CustomerID', 'InvoiceDate']].copy()
customer_months['Month'] = customer_months['InvoiceDate'].dt.to_period('M')
customer_months = customer_months.drop(columns='InvoiceDate').drop_duplicates()

next_month = customer_months.copy()
next_month['Month'] = next_month['Month']+1

retained = pd.merge(customer_months, next_month, on=['CustomerID', 'Month'])

active_per_month = customer_months.groupby('Month')['CustomerID'].count()
retained_per_month = retained.groupby('Month')['CustomerID'].count()

retention_rate = (retained_per_month / active_per_month) * 100

print(customer_months)
print(customer_months.shape)
print(retention_rate)


# 2010 numbers are unreliable - the NaN values appear where there are gaps in months (April and November 2010 are missing).
# The rates of 2-6% reflect the incomplete data.
# Retention starts at 21.6% in January and grows steadily to 42.6% by August.
# There's a dip in September (31.4%) which coincides with the revenue spike.
# December shows 51% but that's a partial month so not fully reliable.
# Retention is actually improving through 2011, which is a positive sign. But even at its best it's only 42%.
# More than half of active customers in any given month don't come back the following month.
# The rates of 2-6% reflect the incomplete data.
# Retention starts at 21.6% in January and grows steadily to 42.6% by August.
# There's a dip in September (31.4%) which coincides with the revenue spike.
# December shows 51% but that's a partial month so not fully reliable.
# Retention is actually improving through 2011, which is a positive sign. But even at its best it's only 42%.
# More than half of active customers in any given month don't come back the following month.


# Step 1 -- Extract relevant columns and make a clean copy
  # Pull only CustomerID and InvoiceDate from df into a new dataframe called customer_months. Using .copy() ensures it's independent from df.

# Step 2 -- Extract the month from InvoiceDate
  # Create a new Month column by converting InvoiceDate to a year-month period using dt.to_period("M"). This groups all transactions within the same month together.

# Step 3 -- Remove duplicates
  # Drop InvoiceDate since it's no longer needed, then remove duplicate rows. This leaves one row per unique customer-month combination -- meaning each customer appears only once per month regardless of how many orders they placed.

# Step 4 -- Create the "next month" table
  # Make a copy of customer_months called next_month and add 1 to every month value. This shifts the entire table forward by one month, representing where each customer "should be" if they were retained.

# Step 5 -- Merge to find retained customers
  # Merge customer_months with next_month on both CustomerID and Month. The merge only keeps rows where the same customer appears in both tables for the same month value -- meaning they were active in month X and also appeared in the next month table's month X, which corresponds to being active in month X-1 of the original data. These are your retained customers.

# Step 6 -- Count active and retained customers per month
  # Group customer_months by Month and count unique customers to get active_per_month. Do the same for retained to get retained_per_month.

# Step 7 -- Calculate retention rate
  # Divide retained_per_month by active_per_month and multiply by 100 to get the percentage of customers retained month over month.

       CustomerID    Month
0           17850  2010-01
9           13047  2010-01
26          12583  2010-01
46          13748  2010-01
65          15100  2010-01
...           ...      ...
541569      15910  2011-09
541589      17754  2011-09
541718      12662  2011-09
541755      12526  2011-09
541768      12713  2011-09

[13016 rows x 2 columns]
(13016, 2)
Month
2010-01          NaN
2010-02     6.060606
2010-03     2.000000
2010-05          NaN
2010-06     3.658537
2010-07     4.615385
2010-08     2.020202
2010-09     2.222222
2010-10     5.660377
2010-12          NaN
2011-01    21.600000
2011-02    33.498759
2011-03    30.434783
2011-04    34.719101
2011-05    35.019841
2011-06    35.140368
2011-07    37.697307
2011-08    42.617801
2011-09    31.370900
2011-10    35.148118
2011-11    37.618403
2011-12    51.069519
Freq: M, Name: CustomerID, dtype: float64


In [20]:
#Q13 - What does the RFM segmentation look like? (Who are the Champions, Loyal, At Risk, and Lost customers?)

rfm = df.groupby('CustomerID').agg(
    Recency = ('InvoiceDate','max'),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary = ('Revenue', 'sum')
)

print(rfm)

reference_date = df['InvoiceDate'].max()
rfm['Recency'] = reference_date - rfm['Recency']
rfm['Recency'] = rfm['Recency'].dt.days

rfm['Segment'] = 'Other'
rfm.loc[(rfm['Recency'] <= 60) & (rfm['Frequency'] > 5) & (rfm['Monetary'] > 1000), 'Segment'] = 'Champion'
rfm.loc[(rfm['Recency'] <= 60) & (rfm['Frequency'] > 5) & (rfm['Monetary'] <= 1000), 'Segment'] = 'Loyal'
rfm.loc[(rfm['Recency'] > 60) & (rfm['Recency'] <= 180), 'Segment'] = 'At Risk'
rfm.loc[(rfm['Recency'] >= 180), 'Segment'] = 'Lost'

rfm['Segment'].value_counts()



# Segment breakdown:
  # Other (1,424) - recent customers with low frequency (5 or fewer orders). Not yet loyal but still active
  # At Risk (1,216) - haven't bought in 2-6 months. Large group worth targeting
  # Lost (958) - haven't bought in over 6 months. Difficult but not impossible to win back
  # Champion (712) - recent, frequent, high spending. Your most valuable customers
  # Loyal (28) - recent and frequent but lower spend. Very small group

# Champions (712 customers) are likely driving the bulk of that 93% repeat revenue -- protect them at all costs
# At Risk (1,216) is the most important group to act on -- they were active but are slipping. A targeted discount or re-engagement email could recover a significant portion
# Lost (958) combined with At Risk means roughly half your customer base needs attention
# The tiny Loyal segment (28) suggests most frequent buyers also spend heavily -- there's little middle ground between Champions and Others

                       Recency  Frequency  Monetary
CustomerID                                         
12346      2011-01-18 10:01:00          1  77183.60
12347      2011-10-31 12:25:00          7   4310.00
12348      2011-09-25 13:13:00          4   1797.24
12349      2011-11-21 09:51:00          1   1757.55
12350      2011-02-02 16:01:00          1    334.40
...                        ...        ...       ...
18280      2011-07-03 09:52:00          1    180.60
18281      2011-12-06 10:53:00          1     80.82
18282      2011-05-08 13:35:00          2    178.05
18283      2011-11-30 12:59:00         16   2094.88
18287      2011-12-10 10:23:00          3   1837.28

[4338 rows x 3 columns]


,count
Segment,
Other,1424
At Risk,1216
Lost,958
Champion,712
Loyal,28


In [21]:
#Q14 - What are the top selling products in the top 5 countries?

df_international = df[df["Country"] != "United Kingdom"]
top5_countries = df_international.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(5).index

df_top5 = df_international[df_international['Country'].isin(top5_countries)]

product_by_country = df_top5.groupby(['Country','Description'])['Revenue'].sum().sort_values(ascending=False)

print(product_by_country['Netherlands'].head(5))
print(product_by_country['EIRE'].head(5))
print(product_by_country['Germany'].head(5))
print(product_by_country['France'].head(5))
print(product_by_country['Australia'].head(5))

# POSTAGE and Manual appear in Germany, France and EIRE -- flag these as administrative entries, not real products
# RABBIT NIGHT LIGHT appears in Netherlands, France and Australia -- this is a genuinely popular international product
# REGENCY CAKESTAND 3 TIER appears in EIRE, Germany, France and Australia -- the most consistently popular product across markets
# ROUND SNACK BOXES dominate Netherlands specifically -- could indicate a market preference worth investigating
# Netherlands and Australia have cleaner results with actual products at the top, making them more useful for decision making

# REGENCY CAKESTAND and RABBIT NIGHT LIGHT are the popular international products, they sell across multiple markets.

Description
RABBIT NIGHT LIGHT                     9568.48
ROUND SNACK BOXES SET OF4 WOODLAND     7991.40
SPACEBOY LUNCH BOX                     7485.60
DOLLY GIRL LUNCH BOX                   6828.60
ROUND SNACK BOXES SET OF 4 FRUITS      4039.20
Name: Revenue, dtype: float64
Description
REGENCY CAKESTAND 3 TIER         7388.55
CARRIAGE                         4875.00
3 TIER CAKE TIN RED AND CREAM    4235.65
Manual                           3374.34
JAM MAKING SET WITH JARS         2976.00
Name: Revenue, dtype: float64
Description
POSTAGE                                21001.00
REGENCY CAKESTAND 3 TIER                9061.95
ROUND SNACK BOXES SET OF4 WOODLAND      3598.95
Manual                                  2296.25
ROUND SNACK BOXES SET OF 4 FRUITS       1982.40
Name: Revenue, dtype: float64
Description
POSTAGE                          15454.00
Manual                            9492.37
RABBIT NIGHT LIGHT                7234.24
REGENCY CAKESTAND 3 TIER          2816.85
RED TOADSTOOL 

In [22]:
#Q15 - Which products drive the most revenue but are only bought once -- high value, low repeat?

product_stats = df.groupby('Description').agg({'Revenue': 'sum', 'CustomerID': 'nunique'})

product_stats[product_stats['CustomerID'] == 1]['Revenue'].sort_values(ascending=False).head(10)


# PAPER CRAFT, LITTLE BIRDIE - £168,469 in revenue from a single customer.
  # That's your top revenue product overall, but it's entirely dependent on one buyer. If that customer leaves, all that revenue disappears overnight
# PICNIC BASKET WICKER 60 PIECES at £39,619 - same story, one customer driving all the revenue
# DOTCOM POSTAGE is likely another administrative entry

# These are high-risk revenue products. These customers are in champion segment and need to be protected.
# Alternative, the product team can develop similar products for the customer or the sales team can pitch other such similar products as well.

,Revenue
Description,
"PAPER CRAFT , LITTLE BIRDIE",168469.60
PICNIC BASKET WICKER 60 PIECES,39619.50
DOTCOM POSTAGE,11906.36
TEA TIME TEA TOWELS,6045.00
POTTING SHED CANDLE CITRONELLA,1250.82
MISELTOE HEART WREATH CREAM,996.00
SET/5 RED SPOTTY LID GLASS BOWLS,734.40
WEEKEND BAG VINTAGE ROSE PAISLEY,527.85
LUNCH BAG RED SPOTTY,290.00


In [23]:
#Q16 - What percentage of revenue comes from the top 20% of customers?

total_monetary_sum = rfm['Monetary'].sum()
top20_revenue_share = ((rfm['Monetary'].sort_values(ascending=False).head(868).sum()) / total_monetary_sum) * 100

top20_revenue_share


# The top 20% of customers generate 74.6% of revenue
# The business is somewhat concentrated but not extremely so -- revenue is spread a little more evenly than the typical 80/20 pattern
# Combined with your earlier finding that 93% of revenue comes from repeat customers, this paints a clear picture -- a relatively small, loyal customer base is holding up the entire business
# If the top 868 customers were lost, nearly 75% of revenue would disappear

# This is a significant business risk. A manager seeing this would want to know what retention strategies exist for the top 20%. Are there account managers assigned to these customers? Are they being offered loyalty incentives?

np.float64(74.61713718676613)

In [24]:
new_customers_monthly.to_frame().to_csv("new_customers_monthly.csv")
retention_rate.to_frame().to_csv("retention_rate.csv")
rfm.to_csv("rfm.csv")

In [25]:
new_customers_monthly.to_frame(name='New_Customers').to_csv("new_customers_monthly.csv")
retention_rate.to_frame(name='Retention_Rate').to_csv("retention_rate.csv")

In [26]:
revenue_concentration = pd.DataFrame({
    'Customer_Group': ['Top 20%', 'Bottom 80%'],
    'Revenue_Share': [74.62, 25.38]
})

revenue_concentration.to_csv("revenue_concentration.csv", index=False)

In [27]:
product_by_country.reset_index().to_csv("product_by_country.csv", index=False)

In [28]:
single_customer_products = product_stats[product_stats['CustomerID'] == 1].sort_values('Revenue', ascending=False).head(10)
single_customer_products.reset_index().to_csv("single_customer_products.csv", index=False)